# Discrete Event Simulation

This is a Store class for a general store

In [ ]:
from sequence.kernel.timeline import Timeline

class Store(object):
    def __init__(self, tl: Timeline):
        self.opening = False
        self.timeline = tl

    def open(self) -> None:
        self.opening = True

    def close(self) -> None:
        self.opening = False

### Scheduling the Events of the store

In [2]:
from sequence.kernel.event import Event
from sequence.kernel.process import Process

tl = Timeline() # create timeline
tl.show_progress = False # turn of progress bar, we will address this in later tutorials.
store = Store(tl) # create store

# open store at 7:00
open_proc = Process(store, 'open', []) # Process(object, function name: string, arguments of function: List[])
open_event = Event(7, open_proc) # Event(occurring time: int, process: Process)
tl.schedule(open_event) # Timeline.schedule(Event)

In [3]:
print(store.opening) # False

False


Run the simulation

In [4]:
tl.run()
print(tl.now(), store.opening) # 7 True

7 True


In [6]:
# open closes at 19:00
close_proc = Process(store, 'close', []) # Process(object, function name: string, arguments of function: List[])
close_event = Event(19, close_proc) # Event(occurring time: int, process: Process)
tl.schedule(close_event) # Timeline.schedule(Event)

tl.run()
print(tl.now(), store.opening) # 7 True

19 False


In [7]:
tl.time = 0
tl.schedule(close_event)
tl.schedule(open_event)
tl.run()
print(tl.time, store.opening)

19 False


In [8]:
tl.time = 0
tl.schedule(open_event)
tl.schedule(close_event)
tl.run()
print(tl.time, store.opening)

19 False


The events work as scheduled when we run it, irrespective of the order, just depends on time

### Opening and closing automatically

In [9]:
from sequence.kernel.timeline import Timeline
from sequence.kernel.event import Event
from sequence.kernel.process import Process

class Store(object):
    def __init__(self, tl: Timeline):
        self.opening = False
        self.timeline = tl

    def open(self) -> None:
        self.opening = True
        process = Process(self, 'close', [])
        event = Event(self.timeline.now() + 12, process)
        self.timeline.schedule(event)

    def close(self) -> None:
        self.opening = False
        process = Process(self, 'open', [])
        event = Event(self.timeline.now() + 12, process)
        self.timeline.schedule(event)

In [11]:
tl = Timeline(60)
tl.show_progress = False
store = Store(tl)
process = Process(store, 'open', [])
event = Event(7, process)
tl.schedule(event)

In [13]:
for t in [15, 32, 52]:
    tl = Timeline(t)
    store = Store(tl)
    print(tl.now())
    
    process = Process(store, 'open', [])
    event = Event(7, process)
    tl.schedule(event)
    
    tl.run()
    print(tl.now())
    print(store.opening)


0
7
True
0
31
True
0
43
False


In [ ]:
from sequence.kernel.timeline import Timeline
from sequence.kernel.event import Event
from sequence.kernel.process import Process
import sequence.utils.log as log

class Store:
    def __init__(self, tl: Timeline):
        self.opening = False
        self.timeline = tl

    def open(self) -> None:
        if self.timeline.now() >= 60:
            self.timeline.stop()
        
        log.logger.info('Store being opened.')
        if self.opening == True:
            log.logger.warning('Store was already open.')

        self.opening = True
        process = Process(self, 'close', [])
        event = Event(self.timeline.now() + 12, process)
        self.timeline.schedule(event)

    def close(self) -> None:
        if self.timeline.now() >= 60:
            self.timeline.stop()

        log.logger.info('Store being closed.')
        if self.opening == False:
            log.logger.warning('Store was already closed.')

        self.opening = False
        process = Process(self, 'open', [])
        event = Event(self.timeline.now() + 12, process)
        self.timeline.schedule(event)

In [15]:
if __name__ == '__main__':
    tl = Timeline()
    tl.show_progress = False
    store = Store(tl)
    process = Process(store, 'open', [])
    event = Event(7, process)
    tl.schedule(event)
    
    log_filename = 'store.log'
    log.set_logger(__name__, tl, log_filename)
    log.set_logger_level('INFO')
    log.track_module('des')
    tl.run()

### Closing Remarks

Event - Class used for denote a process at a time

Process - Anything to be executed

Timeline - Kernal for executing the processes in a timeline

Logger - Sort of same as python logger